In [ ]:
import json
from pathlib import Path
from functools import lru_cache

In [ ]:
# OUTPUT = "/workspaces/dev/output/libri/clean/rt_whisper_each_with_se47_4.json"
OUTPUT = "/workspaces/dev/output/esic/overall_each.json"

In [ ]:
output = Path(OUTPUT)
output.exists()

In [ ]:
json_output = json.loads(output.read_text())

In [ ]:
METRIC = ["correct_percent", "substitution_percent", "deletion_percent", "insertion_percent", "wer_percent", "sentence_error_percent"]

def transform_data(data: dict) -> list[dict]:
    transformed = []
    for key, value in data.items():
        transformed.append({
            "name": key,
            "value": value,
        })
    return transformed

@lru_cache(maxsize=128)
def get_data(name:str) -> dict:
    data = json_output[name]
    del data["processed_time"]
    del data["transcribe_time"]
    validate_metric(data)
    data = transform_data(data)
    return data

def validate_metric(data: dict):
    for key, value in data.items():
        for m in METRIC:
            if value.get(m, None) is None:
                raise ValueError(f"Metric {m} is missing in {key}")

def metric_view(max_name: str, max_val: float, min_name: str, min_val: float, metric_name: str):
    return f"\t{'max:':<8} {max_name:<20} ({max_val:.4f})  -  {'min:':<8} {min_name:<20} ({min_val:.4f})"


def sorted_all_metric(data:list[dict]):
    result = {}
    for i in range(len(METRIC)):
        metric = METRIC[i]
        sorted_data = sorted(data, key=lambda x: x["value"][metric], reverse=True)
        result[metric] = sorted_data
    return result

def analyze_all_metric(sorted_dict:dict[list], k:int = 1):
    for i in range(len(METRIC)):
        metric = METRIC[i]
        sorted_data = sorted_dict[metric]
        max_name = sorted_data[0]['name']
        max_val = sorted_data[0]['value'][metric]
        min_name = sorted_data[-1]['name']
        min_val = sorted_data[-1]['value'][metric]

        print(f"{metric}:")
        print(metric_view(max_name, max_val, min_name, min_val, metric))
        for j in range(1, k):
            max_name = sorted_data[j]['name']
            max_val = sorted_data[j]['value'][metric]
            min_name = sorted_data[-(j+1)]['name']
            min_val = sorted_data[-(j+1)]['value'][metric]
            print(metric_view(max_name, max_val, min_name, min_val, metric))

def analyze_frequency(sorted_data:dict[list], k:int = 1):
    frequency = {}
    for metric in METRIC:
        if metric == "correct_percent":
            for d in sorted_data[metric][-k:]:
                if d['name'] not in frequency:
                    frequency[d['name']] = 0
                frequency[d['name']] += 1
        else:
            for d in sorted_data[metric][:k]:
                if d['name'] not in frequency:
                    frequency[d['name']] = 0
                frequency[d['name']] += 1

    sorted_frequency = sorted(frequency.items(), key=lambda x: x[1], reverse=True)
    print(f"\nFrequency of top {k} metrics:")
    for name, freq in sorted_frequency:
        print(f"{name}: {freq}")
    print("\nAnalysis complete.")

In [ ]:
rt_whisper = get_data("rt_whisper")
sorted_data = sorted_all_metric(rt_whisper)

In [ ]:
analyze_all_metric(sorted_data, 4)

In [ ]:
analyze_frequency(sorted_data, 3)